# 02 - EDA + Phân tích thống kê + Quan hệ đặc trưng + Trực quan hóa
**Người 2 — Nhánh:** `feature/visualization`

**Đầu vào:** `data/processed/` (do Người 1 tạo ra)  
**Biến mục tiêu:** `formatted_experience_level`

## 0. Import thư viện & Cài đặt chung

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, f_oneway
from pathlib import Path
import warnings

# Tắt các cảnh báo không cần thiết
warnings.filterwarnings('ignore')

# Cài đặt hiển thị bảng
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Cài đặt đồ thị
sns.set_style('whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Đường dẫn thư mục dữ liệu đã xử lý
PROCESSED = Path('../data/processed')

print('Import thư viện thành công!')

## 1. Tải dữ liệu

In [ ]:
# Tải bảng chính: postings_clean
df = pd.read_csv(PROCESSED / 'postings_clean.csv', low_memory=False)
print(f'postings_clean: {df.shape[0]:,} dòng x {df.shape[1]} cột')

# Hàm tải bảng phụ, kiểm tra tồn tại trước khi tải
def load(filename):
    path = PROCESSED / filename
    if path.exists():
        data = pd.read_csv(path)
        print(f'{filename}: {data.shape[0]:,} dòng x {data.shape[1]} cột')
        return data
    print(f'Không tìm thấy file: {filename}')
    return None

job_skills     = load('job_skills_clean.csv')
companies      = load('companies_clean.csv')
salaries       = load('salaries_clean.csv')
job_industries = load('job_industries_clean.csv')

In [ ]:
# Xem tổng quan dữ liệu
print('Danh sách cột trong postings_clean:')
print(list(df.columns))
df.head()

In [ ]:
# Xác định nhóm cột để phân tích
TARGET = 'formatted_experience_level'

# Cột số: lương và mức độ tương tác
NUM_COLS = [c for c in ['min_salary', 'med_salary', 'max_salary',
                         'normalized_salary', 'views', 'applies']
            if c in df.columns]

# Cột phân loại
CAT_COLS = [c for c in ['formatted_work_type', 'work_type', 'remote_allowed',
                          'pay_period', 'compensation_type', 'currency']
            if c in df.columns]

# Cột văn bản
TEXT_COLS = [c for c in ['title', 'description', 'skills_desc'] if c in df.columns]

print('Biến mục tiêu :', TARGET,
      '| Thiếu:', df[TARGET].isnull().sum(),
      f'({df[TARGET].isnull().mean()*100:.1f}%)')
print('Cột số        :', NUM_COLS)
print('Cột phân loại :', CAT_COLS)
print('Cột văn bản   :', TEXT_COLS)

In [ ]:
# Kiểm tra giá trị thiếu (chỉ hiển thị cột có null)
missing = pd.DataFrame({
    'Kiểu dữ liệu'         : df.dtypes,
    'Số null'              : df.isnull().sum(),
    '% null'               : (df.isnull().sum() / len(df) * 100).round(2),
    'Số giá trị khác nhau': df.nunique()
})
missing[missing['Số null'] > 0].sort_values('% null', ascending=False)

## 2. Phân tích đơn biến — Cột số (Univariate Numerical)
Phân tích từng cột số: Trung bình, Trung vị, Độ lệch chuẩn, IQR, Độ lệch (Skewness), Ngoại lệ  
Biểu đồ: Histogram + KDE + Boxplot

In [ ]:
# Thống kê mô tả đầy đủ cho các cột số
print('=== THỐNG KÊ MÔ TẢ CÁC CỘT SỐ ===')
desc = df[NUM_COLS].describe().T
desc['IQR']            = desc['75%'] - desc['25%']
desc['Độ lệch (skew)'] = df[NUM_COLS].skew()
desc['Kurtosis']       = df[NUM_COLS].kurtosis()
desc.rename(columns={'25%': 'Q1', '50%': 'Trung vị', '75%': 'Q3'}, inplace=True)
desc.round(2)

In [ ]:
# Vẽ Histogram + KDE + Boxplot cho từng cột số
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        print(f'Cột {col}: toàn bộ là NaN, bỏ qua.')
        continue

    # Tính các chỉ số phân vị và ngoại lệ theo phương pháp IQR
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR   # Ngưỡng dưới
    upper  = Q3 + 1.5 * IQR   # Ngưỡng trên
    n_out  = ((data < lower) | (data > upper)).sum()   # Số ngoại lệ

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # --- Biểu đồ Histogram + KDE ---
    axes[0].hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.7, density=True)
    data.plot.kde(ax=axes[0], color='crimson', linewidth=2)
    axes[0].axvline(data.mean(),   color='orange', linestyle='--', linewidth=1.5,
                    label=f'Trung bình = {data.mean():,.0f}')
    axes[0].axvline(data.median(), color='green',  linestyle='--', linewidth=1.5,
                    label=f'Trung vị   = {data.median():,.0f}')
    axes[0].set_title(f'Phân phối — {col}', fontsize=13, fontweight='bold')
    axes[0].legend()

    # Nhận xét độ lệch phân phối
    skew = data.skew()
    if skew > 0.5:
        skew_txt = 'Lệch phải (dương)'
    elif skew < -0.5:
        skew_txt = 'Lệch trái (âm)'
    else:
        skew_txt = 'Đối xứng'
    axes[0].text(0.97, 0.95, f'Skew = {skew:.2f}\n({skew_txt})',
                 transform=axes[0].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    # --- Biểu đồ Boxplot ---
    axes[1].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.3, markersize=3))
    axes[1].set_title(f'Boxplot — {col}', fontsize=13, fontweight='bold')
    axes[1].text(0.97, 0.85,
                 f'Q1 = {Q1:,.0f}\nQ3 = {Q3:,.0f}\nIQR = {IQR:,.0f}\n'
                 f'Ngoại lệ: {n_out:,} ({n_out/len(data)*100:.1f}%)',
                 transform=axes[1].transAxes, ha='right', va='top', fontsize=9,
                 bbox=dict(facecolor='lightyellow', alpha=0.9, boxstyle='round'))

    plt.suptitle(f'Phân tích đơn biến: {col}  (n={len(data):,}, thiếu={df[col].isnull().sum():,})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Phân tích đơn biến — Cột phân loại (Univariate Categorical)
Tần suất, Tỷ lệ phần trăm, Cardinality  
Biểu đồ: Cột đếm + Cột tỷ lệ

In [ ]:
# Bảng tần suất cho từng cột phân loại
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False)
    pct = df[col].value_counts(normalize=True, dropna=False) * 100
    # Chuyển đổi nhãn hiển thị, thay thế NaN thành '(Trống)'
    lbls = [str(x) if pd.notna(x) else '(Trống)' for x in vc.index]
    tbl = pd.DataFrame({'Giá trị': lbls, 'Số lượng': vc.values, 'Tỷ lệ (%)': pct.round(2).values})
    print(f'\n{col}  | Cardinality={df[col].nunique()} | '
          f'Thiếu={df[col].isnull().sum():,} ({df[col].isnull().mean()*100:.1f}%)')
    print(tbl.to_string(index=False))

In [ ]:
# Vẽ biểu đồ cột cho từng cột phân loại (xử lý an toàn nhãn NaN)
for col in CAT_COLS:
    vc  = df[col].value_counts(dropna=False).head(15)
    pct = vc / len(df) * 100

    # Chuyển nhãn thành danh sách chuỗi (str) thuần túy để tránh lỗi kiểu dữ liệu
    x_labels = [str(x) if pd.notna(x) else '(Trống)' for x in vc.index]
    colors   = sns.color_palette('Set2', len(vc))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Biểu đồ số lượng
    bars = axes[0].bar(x_labels, vc.values, color=colors, edgecolor='white')
    for bar, p in zip(bars, pct.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + vc.values.max() * 0.01,
                     f'{p:.1f}%', ha='center', va='bottom', fontsize=9)
    axes[0].set_title(f'Số lượng — {col}', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=40)

    # Biểu đồ tỷ lệ phần trăm
    axes[1].bar(x_labels, pct.values, color=colors, edgecolor='white')
    axes[1].set_ylabel('Tỷ lệ (%)')
    axes[1].set_title(f'Tỷ lệ phần trăm — {col}', fontsize=12, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=40)

    # Đường cân bằng lý tưởng (nếu dữ liệu phân bổ đều giữa các nhóm)
    n_cat = df[col].nunique()
    if n_cat > 0:
        axes[1].axhline(100 / n_cat, color='red', linestyle='--', linewidth=1,
                        label=f'Cân bằng lý tưởng = {100/n_cat:.1f}%')
        axes[1].legend(fontsize=8)

    plt.suptitle(f'Phân tích đơn biến: {col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Quan hệ Đặc trưng — Đặc trưng (Numerical vs Numerical)
Tương quan Pearson & Spearman, Đa cộng tuyến (Multicollinearity), Pair Plot

In [ ]:
# Tính ma trận tương quan Pearson và Spearman
pearson  = df[NUM_COLS].corr(method='pearson')
spearman = df[NUM_COLS].corr(method='spearman')

# Chỉ hiển thị nửa dưới của ma trận để tránh trùng lặp
mask = np.triu(np.ones_like(pearson, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(pearson, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[0], linewidths=0.5, annot_kws={'size': 10})
axes[0].set_title('Tương quan Pearson', fontsize=13, fontweight='bold')

sns.heatmap(spearman, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            ax=axes[1], linewidths=0.5, annot_kws={'size': 10})
axes[1].set_title('Tương quan Spearman', fontsize=13, fontweight='bold')

plt.suptitle('Ma trận tương quan giữa các đặc trưng số', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bảng phân loại mức độ tương quan từng cặp
rows = []
for i in range(len(NUM_COLS)):
    for j in range(i + 1, len(NUM_COLS)):
        r  = pearson.iloc[i, j]
        rs = spearman.iloc[i, j]
        a  = abs(r)
        # Phân loại mức độ tương quan
        if a > 0.7:
            strength = 'Mạnh (>0.7)'
        elif a > 0.4:
            strength = 'Trung bình (0.4–0.7)'
        else:
            strength = 'Yếu (<0.4)'
        direction = 'Dương (+)' if r > 0 else 'Âm (−)'
        rows.append({
            'Đặc trưng 1' : NUM_COLS[i],
            'Đặc trưng 2' : NUM_COLS[j],
            'Pearson r'   : round(r, 3),
            'Spearman r'  : round(rs, 3),
            'Mức độ'      : strength,
            'Chiều'       : direction
        })

corr_df = pd.DataFrame(rows).sort_values('Pearson r', key=abs, ascending=False)
print('=== PHÂN LOẠI MỨC ĐỘ TƯƠNG QUAN ===')
print(corr_df.to_string(index=False))

# Kiểm tra đa cộng tuyến
print('\n--- ĐA CỘNG TUYẾN — MULTICOLLINEARITY (|r| > 0.7) ---')
multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
if len(multi):
    print(multi[['Đặc trưng 1', 'Đặc trưng 2', 'Pearson r']].to_string(index=False))
    print('=> Các cặp này mang thông tin trùng lặp, cân nhắc loại bỏ khi modeling.')
else:
    print('Không phát hiện đa cộng tuyến nghiêm trọng.')

In [ ]:
# Vẽ Pair Plot để quan sát quan hệ từng cặp đặc trưng số
# Chọn các cột có số lượng dòng quan sát chung đủ lớn
pair_cols = [c for c in ['normalized_salary', 'views', 'applies'] if c in df.columns]
sample_df = df[pair_cols].dropna()

if len(sample_df) > 0:
    n_sample = min(2000, len(sample_df))
    sample = sample_df.sample(n_sample, random_state=42)
    g = sns.pairplot(sample, diag_kind='kde',
                     plot_kws={'alpha': 0.25, 's': 15, 'color': 'steelblue'},
                     diag_kws={'color': 'steelblue', 'fill': True})
    g.fig.suptitle(f'Pair Plot — Các đặc trưng số (mẫu {n_sample:,} dòng)',
                   fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Không có đủ dữ liệu đồng thời giữa các cột số để vẽ Pair Plot.')

## 5. Quan hệ Đặc trưng — Mục tiêu (Numerical vs Target)
So sánh các cột số theo từng cấp độ kinh nghiệm  
Kiểm định thống kê: ANOVA và Kruskal-Wallis  
Biểu đồ: Boxplot, Violin Plot

In [ ]:
# Thống kê mô tả từng cột số theo cấp độ kinh nghiệm
if TARGET in df.columns:
    for col in NUM_COLS:
        tbl = df.groupby(TARGET)[col].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        print(f'\n=== {col.upper()} theo {TARGET} ===')
        print(tbl.round(2))

In [ ]:
# Vẽ Boxplot và Violin Plot theo từng Job Level
if TARGET in df.columns:
    for col in NUM_COLS:
        plot_df = df[[TARGET, col]].dropna()
        if len(plot_df) == 0:
            continue

        # Sắp xếp Job Level theo trung vị giảm dần
        order = plot_df.groupby(TARGET)[col].median().sort_values(ascending=False).index

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # Boxplot
        sns.boxplot(data=plot_df, x=TARGET, y=col,
                    order=order, palette='Set2', ax=axes[0])
        axes[0].set_title(f'{col} theo Cấp độ — Boxplot', fontsize=12, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=30)

        # Violin Plot (thể hiện hình dạng phân phối)
        sns.violinplot(data=plot_df, x=TARGET, y=col, order=order,
                       palette='Set2', ax=axes[1], inner='quartile', cut=0)
        axes[1].set_title(f'{col} theo Cấp độ — Violin', fontsize=12, fontweight='bold')
        axes[1].tick_params(axis='x', rotation=30)

        plt.suptitle(f'Quan hệ Đặc trưng–Mục tiêu: {col} vs {TARGET}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
# Kiểm định thống kê: ANOVA và Kruskal-Wallis
if TARGET in df.columns:
    print('=== KIỂM ĐỊNH THỐNG KÊ (α = 0.05) ===')
    print('p < 0.05  →  đặc trưng có quan hệ có ý nghĩa với biến mục tiêu\n')

    results = []
    for col in NUM_COLS:
        # Tách dữ liệu thành từng nhóm theo Job Level
        groups = [df[df[TARGET] == g][col].dropna().values
                  for g in df[TARGET].dropna().unique()]
        groups = [g for g in groups if len(g) > 1]
        if len(groups) < 2:
            continue

        f_val, p_anova = f_oneway(*groups)   # Kiểm định ANOVA (giả định phân phối chuẩn)
        h_val, p_kw    = kruskal(*groups)    # Kiểm định Kruskal-Wallis (không tham số)

        results.append({
            'Đặc trưng'       : col,
            'ANOVA F'         : round(f_val, 2),
            'ANOVA p'         : round(p_anova, 5),
            'Kruskal-Wallis H': round(h_val, 2),
            'KW p'            : round(p_kw, 5),
            'Kết luận'        : 'Có quan hệ' if p_kw < 0.05 else 'Không rõ ràng'
        })

    print(pd.DataFrame(results).to_string(index=False))
    print('\nLưu ý: Ưu tiên Kruskal-Wallis vì dữ liệu salary/views bị lệch mạnh (skewed).')

## 6. Phân tích Phân loại — Số (Categorical vs Numerical)
Lương, Lượt xem, Lượt ứng tuyển theo Loại công việc, Làm từ xa, Chu kỳ thanh toán

In [ ]:
# Chọn cột lương ưu tiên nhất có sẵn trong dữ liệu
sal_col = next((c for c in ['normalized_salary', 'med_salary', 'min_salary']
                if c in df.columns), None)

# Tạo danh sách các cặp (cột phân loại, cột số) cần phân tích
pairs = []
for cat in ['formatted_work_type', 'remote_allowed', 'pay_period']:
    if cat in df.columns and sal_col:
        pairs.append((cat, sal_col))   # Lương theo từng loại
for num in ['views', 'applies']:
    if 'formatted_work_type' in df.columns and num in df.columns:
        pairs.append(('formatted_work_type', num))   # Lượt xem/ứng tuyển theo loại công việc

# Vẽ Boxplot và biểu đồ trung bình
for cat_col, num_col in pairs:
    plot_df = df[[cat_col, num_col]].dropna()
    if len(plot_df) == 0:
        continue

    # Sắp xếp theo trung vị giảm dần
    order = plot_df.groupby(cat_col)[num_col].median().sort_values(ascending=False).index

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Boxplot
    sns.boxplot(data=plot_df, x=cat_col, y=num_col,
                order=order, palette='Set2', ax=axes[0])
    axes[0].set_title(f'{num_col} theo {cat_col} — Boxplot', fontsize=11, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Biểu đồ cột trung bình
    means = plot_df.groupby(cat_col)[num_col].mean().reindex(order)
    x_mean_labels = [str(x) for x in means.index]
    bars  = axes[1].bar(x_mean_labels, means.values,
                         color=sns.color_palette('Set2', len(means)), edgecolor='white')
    for bar, v in zip(bars, means.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height(), f'{v:,.0f}',
                     ha='center', va='bottom', fontsize=9)
    axes[1].set_title(f'Trung bình {num_col} theo {cat_col}', fontsize=11, fontweight='bold')
    axes[1].tick_params(axis='x', rotation=30)

    plt.suptitle(f'Phân loại vs Số: {cat_col} × {num_col}', fontsize=13)
    plt.tight_layout()
    plt.show()

## 7. Phân tích Phân loại — Phân loại (Categorical vs Categorical)
Bảng chéo (Crosstab), Kiểm định Chi-square, Biểu đồ cột chồng (Stacked Bar)

In [ ]:
if TARGET in df.columns:
    # Danh sách cột phân loại cần so sánh với biến mục tiêu
    cat_list     = [c for c in ['formatted_work_type', 'remote_allowed', 'pay_period']
                    if c in df.columns]
    chi2_results = []

    for col2 in cat_list:
        plot_df = df[[TARGET, col2]].dropna()
        if len(plot_df) == 0:
            continue

        # Tạo bảng chéo số lượng và tỷ lệ phần trăm
        ct     = pd.crosstab(plot_df[TARGET], plot_df[col2])
        ct_pct = pd.crosstab(plot_df[TARGET], plot_df[col2], normalize='index') * 100

        print(f'\n=== BẢNG CHÉO: {TARGET} × {col2} ===')
        print(ct)
        print('\nTỷ lệ % theo hàng:')
        print(ct_pct.round(1))

        # Kiểm định Chi-square: kiểm tra sự độc lập giữa 2 biến phân loại
        chi2, p, dof, _ = chi2_contingency(ct)
        chi2_results.append({
            'Cặp biến'  : f'{TARGET} × {col2}',
            'Chi2'      : round(chi2, 2),
            'p-value'   : round(p, 5),
            'Bậc tự do' : dof,
            'Ý nghĩa?'  : 'CÓ' if p < 0.05 else 'KHÔNG'
        })

        # Biểu đồ cột chồng theo tỷ lệ
        ct_pct.plot(kind='bar', stacked=True, figsize=(12, 5),
                    colormap='Set2', edgecolor='white', linewidth=0.5)
        plt.title(f'Biểu đồ cột chồng: {TARGET} × {col2}'
                  f'\n(Chi2={chi2:.1f}, p={p:.4f})',
                  fontsize=12, fontweight='bold')
        plt.ylabel('Tỷ lệ (%)')
        plt.xticks(rotation=30)
        plt.legend(title=col2, bbox_to_anchor=(1.05, 1))
        plt.tight_layout()
        plt.show()

    print('\n=== TÓM TẮT KIỂM ĐỊNH CHI-SQUARE ===')
    print(pd.DataFrame(chi2_results).to_string(index=False))

## 8. Phân tích Đa biến (Multivariate Analysis)
Lương × Cấp độ × Loại công việc + Heatmap tương quan toàn bộ

In [ ]:
# Phân tích 3 chiều: Lương × Cấp độ kinh nghiệm × Loại công việc
sal_col  = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
work_col = 'formatted_work_type'

if TARGET in df.columns and sal_col and work_col in df.columns:
    plot_df = df[[TARGET, sal_col, work_col]].dropna()
    order   = plot_df.groupby(TARGET)[sal_col].median().sort_values().index

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=plot_df, x=TARGET, y=sal_col,
                hue=work_col, order=order, palette='Set2', ax=ax)
    ax.set_title(f'Phân tích đa biến: {sal_col} × {TARGET} × {work_col}',
                 fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Loại công việc', bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap tương quan đầy đủ: cột số + mã hóa số cột phân loại
df_enc = df[NUM_COLS].copy()
for col in CAT_COLS:
    if col in df.columns and df[col].nunique() <= 10:
        # Mã hóa số cho cột phân loại có ít nhãn (≤ 10 giá trị)
        df_enc[col + '_code'] = df[col].astype('category').cat.codes

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(df_enc.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.3, ax=ax, annot_kws={'size': 8})
ax.set_title('Heatmap tương quan đầy đủ (Số + Phân loại đã mã hóa)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Phân tích Ngoại lệ (Outlier Analysis)
Phương pháp IQR & Z-score — Phân biệt lỗi dữ liệu vs giá trị cực trị hợp lệ

In [ ]:
# Bảng tổng hợp ngoại lệ cho tất cả cột số
print('=== PHÂN TÍCH NGOẠI LỆ ===')
outlier_rows = []
for col in NUM_COLS:
    data = df[col].dropna()
    if len(data) == 0:
        continue

    # Phương pháp IQR
    Q1, Q3  = data.quantile(0.25), data.quantile(0.75)
    IQR     = Q3 - Q1
    lower   = Q1 - 1.5 * IQR   # Ngưỡng dưới
    upper   = Q3 + 1.5 * IQR   # Ngưỡng trên
    n_iqr   = ((data < lower) | (data > upper)).sum()

    # Phương pháp Z-score (ngưỡng 3 độ lệch chuẩn)
    n_z = (np.abs(stats.zscore(data)) > 3).sum()

    outlier_rows.append({
        'Đặc trưng'       : col,
        'n'               : len(data),
        'Ngoại lệ IQR'    : n_iqr,
        'IQR %'           : round(n_iqr / len(data) * 100, 2),
        'Ngoại lệ Z>3σ'   : n_z,
        'Z %'             : round(n_z / len(data) * 100, 2),
        'Ngưỡng dưới IQR' : round(lower, 1),
        'Ngưỡng trên IQR' : round(upper, 1)
    })

print(pd.DataFrame(outlier_rows).to_string(index=False))
print('\nNhận xét: Ngoại lệ lương = Giá trị cực trị hợp lệ (lương Director/Executive)')
print('=> Không xóa. Nên dùng Log Transform khi modeling để giảm ảnh hưởng.')

In [ ]:
# So sánh phân phối gốc và sau Log Transform cho cột lương
sal_col = next((c for c in ['normalized_salary', 'min_salary'] if c in df.columns), None)
if sal_col:
    data   = df[sal_col].dropna()
    Q1, Q3 = data.quantile(0.25), data.quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_out  = ((data < lower) | (data > upper)).sum()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Boxplot để thấy vị trí ngoại lệ
    axes[0].boxplot(data, vert=False, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='gray', alpha=0.2, markersize=3))
    axes[0].set_title(f'Boxplot: {sal_col}\n'
                      f'Ngoại lệ = {n_out:,} ({n_out/len(data)*100:.1f}%)',
                      fontsize=11, fontweight='bold')

    # Phân phối gốc (bị lệch mạnh)
    axes[1].hist(data, bins=60, color='steelblue', alpha=0.7, edgecolor='white', density=True)
    axes[1].set_title(f'Phân phối gốc\nĐộ lệch (Skew) = {data.skew():.2f}',
                      fontsize=11, fontweight='bold')

    # Sau Log Transform (phân phối gần chuẩn hơn)
    log_data = np.log1p(data[data > 0])
    axes[2].hist(log_data, bins=60, color='seagreen', alpha=0.7, edgecolor='white', density=True)
    axes[2].set_title(f'Sau Log(1+x)\nĐộ lệch (Skew) = {log_data.skew():.2f}',
                      fontsize=11, fontweight='bold')
    axes[2].set_xlabel('log(1 + lương)')

    plt.suptitle(f'Ngoại lệ & Log Transform — {sal_col}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 10. Phân tích mất cân bằng lớp (Class Imbalance)
Biến mục tiêu: `formatted_experience_level` → Bàn giao kết quả cho Người 4

In [ ]:
if TARGET in df.columns:
    # Đếm số lượng và tỷ lệ từng lớp (kể cả NaN)
    counts = df[TARGET].value_counts(dropna=False)
    pcts   = df[TARGET].value_counts(normalize=True, dropna=False) * 100

    # Chuyển nhãn an toàn, không để NaN dạng float
    target_labels = [str(x) if pd.notna(x) else '(Trống / NaN)' for x in counts.index]

    print('=== PHÂN PHỐI CÁC LỚP — formatted_experience_level ===')
    tbl_imbalance = pd.DataFrame({'Lớp': target_labels, 'Số lượng': counts.values, 'Tỷ lệ (%)': pcts.round(2).values})
    print(tbl_imbalance.to_string(index=False))

    # Tính tỷ lệ mất cân bằng (lớp lớn nhất / lớp bé nhất không tính NaN)
    valid_counts = df[TARGET].dropna().value_counts()
    if len(valid_counts) > 1:
        ratio = valid_counts.max() / valid_counts.min()
        print(f'\nTỷ lệ mất cân bằng (lớp lớn nhất / lớp bé nhất) = {ratio:.1f} : 1')
        if ratio > 5:
            print('MẤT CÂN BẰNG NGHIÊM TRỌNG')
            print('=> Người 4 cần xử lý: SMOTE / class_weight / stratified split')
        elif ratio > 2:
            print('MẤT CÂN BẰNG VỪA')
            print('=> Nên dùng class_weight khi huấn luyện mô hình')
        else:
            print('Tương đối cân bằng')

    colors = sns.color_palette('Set2', len(counts))
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Biểu đồ cột
    bars = axes[0].bar(target_labels, counts.values, color=colors, edgecolor='white')
    for bar, cnt, pct in zip(bars, counts.values, pcts.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + counts.values.max() * 0.01,
                     f'{cnt:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Phân phối lớp — Cấp độ kinh nghiệm', fontsize=13, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=30)

    # Biểu đồ tròn
    axes[1].pie(counts.values,
                labels=[f'{lbl}\n({p:.1f}%)' for lbl, p in zip(target_labels, pcts.values)],
                colors=colors, startangle=90,
                wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
    axes[1].set_title('Tỷ lệ phần trăm — Cấp độ kinh nghiệm', fontsize=13, fontweight='bold')

    plt.suptitle('Phân tích mất cân bằng lớp  →  Bàn giao Người 4',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 11. Phân tích Kỹ năng & Văn bản (Skill & Text Analysis)
Độ dài văn bản, Số từ, Từ khóa phổ biến, Kỹ năng theo Cấp độ

In [ ]:
# Tạo đặc trưng từ cột văn bản
if 'title' in df.columns:
    df['title_length']     = df['title'].fillna('').str.len()              # Độ dài tiêu đề (ký tự)
    df['title_word_count'] = df['title'].fillna('').str.split().str.len()  # Số từ trong tiêu đề
if 'description' in df.columns:
    df['desc_length']      = df['description'].fillna('').str.len()        # Độ dài mô tả
    df['desc_word_count']  = df['description'].fillna('').str.split().str.len()  # Số từ trong mô tả

text_feat = [c for c in ['title_length', 'title_word_count', 'desc_length', 'desc_word_count']
             if c in df.columns]
if text_feat:
    print('=== THỐNG KÊ ĐẶC TRƯNG VĂN BẢN ===')
    print(df[text_feat].describe().round(2))

In [ ]:
# So sánh độ dài văn bản theo từng Cấp độ kinh nghiệm
if TARGET in df.columns:
    for col in ['desc_length', 'title_length']:
        if col not in df.columns:
            continue
        plot_df = df[[TARGET, col]].dropna()
        order   = plot_df.groupby(TARGET)[col].median().sort_values(ascending=False).index

        fig, ax = plt.subplots(figsize=(12, 5))
        sns.boxplot(data=plot_df, x=TARGET, y=col,
                    order=order, palette='Set2', ax=ax)
        ax.set_title(f'{col} theo Cấp độ kinh nghiệm', fontsize=12, fontweight='bold')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()

In [ ]:
# Phân tích từ khóa phổ biến trong mô tả công việc
if 'description' in df.columns:
    # Danh sách từ dừng (stop words) cần loại bỏ
    stop_words = {'the','and','or','in','of','to','a','for','is','are','with','on',
                  'at','be','as','an','we','our','you','will','this','that','have',
                  'it','from','by','was','not','your','can','has','all','they','their',
                  'work','role','team','job','experience','position','skills','ability'}

    # Tách từ, loại stop words, đếm tần suất
    all_words = (df['description'].fillna('')
                   .str.lower()
                   .str.replace(r'[^a-z\s]', ' ', regex=True)
                   .str.split().explode())
    top_keywords = (all_words[(all_words.str.len() > 3) & (~all_words.isin(stop_words))]
                      .value_counts().head(25))

    kw_labels = [str(x) for x in top_keywords.index]
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(kw_labels, top_keywords.values,
           color=sns.color_palette('viridis', 25), edgecolor='white')
    ax.set_title('Top 25 Từ khóa phổ biến trong Mô tả Công việc',
                 fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top kỹ năng và phân tích kỹ năng theo Cấp độ
if job_skills is not None:
    skill_col = next((c for c in ['skill_abr', 'skill_name', 'skill']
                      if c in job_skills.columns), None)
    if skill_col:
        # Top 20 kỹ năng phổ biến nhất trong toàn dataset
        top20 = job_skills[skill_col].value_counts().head(20)
        skill_labels = [str(x) if pd.notna(x) else '(Trống)' for x in top20.index]

        fig, ax = plt.subplots(figsize=(14, 6))
        ax.bar(skill_labels, top20.values,
               color=sns.color_palette('Set2', len(top20)), edgecolor='white')
        ax.set_title('Top 20 Kỹ năng phổ biến nhất (job_skills_clean.csv)',
                     fontsize=13, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

        # Phân tích kỹ năng theo từng Cấp độ kinh nghiệm
        if TARGET in df.columns:
            id_col = next((c for c in ['job_id', 'jobId']
                           if c in job_skills.columns and c in df.columns), None)
            if id_col:
                # Ghép bảng kỹ năng với bảng chính qua job_id
                merged = job_skills.merge(
                    df[[id_col, TARGET]].dropna(), on=id_col, how='inner')
                print('\n=== TOP 5 KỸ NĂNG THEO CẤP ĐỘ KINH NGHIỆM ===')
                for level in sorted(merged[TARGET].unique()):
                    top5 = merged[merged[TARGET] == level][skill_col].value_counts().head(5)
                    print(f'\n  {level}:')
                    print(top5.to_string())

## 12. Nhận xét nghiệp vụ & Dashboard tổng kết (Business Insights)

In [ ]:
print('=' * 65)
print('  NHẬN XÉT NGHIỆP VỤ — DS JOB RECOMMEND')
print('=' * 65)

if TARGET in df.columns:
    counts = df[TARGET].value_counts()
    print(f'\n1. Cấp độ tuyển dụng nhiều nhất: {counts.idxmax()} ({counts.max():,} tin)')
    print(f'   Cấp độ tuyển dụng ít nhất:    {counts.idxmin()} ({counts.min():,} tin)')

sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)
if sal_col and TARGET in df.columns:
    by_level = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False)
    print(f'\n2. Lương trung vị (median) theo cấp độ ({sal_col}):')
    for level, val in by_level.items():
        print(f'   {level}: ${val:,.0f}')

if 'corr_df' in dir() and len(corr_df):
    multi = corr_df[corr_df['Pearson r'].abs() > 0.7]
    print(f'\n3. Đa cộng tuyến (|r| > 0.7):')
    if len(multi):
        for _, row in multi.iterrows():
            print(f'   {row["Đặc trưng 1"]} <-> {row["Đặc trưng 2"]}: r={row["Pearson r"]}')
    else:
        print('   Không phát hiện đa cộng tuyến nghiêm trọng.')

print('\nBàn giao kết quả:')
print('   → Người 3 (Feature Engineering): log_salary, salary_range, text_length features')
print('   → Người 4 (Modeling): tỷ lệ mất cân bằng lớp, danh sách đặc trưng có quan hệ')
print('=' * 65)

In [ ]:
# Dashboard tổng kết EDA — 4 biểu đồ chính
sal_col = next((c for c in ['normalized_salary', 'med_salary'] if c in df.columns), None)

if sal_col and TARGET in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    palette = sns.color_palette('Set2')

    # --- Ô 1: Phân phối cấp độ kinh nghiệm (bỏ qua NaN để biểu đồ rõ ràng) ---
    valid_target = df[TARGET].dropna().value_counts()
    target_x = [str(x) for x in valid_target.index]
    axes[0, 0].bar(target_x, valid_target.values,
                   color=palette[:len(valid_target)], edgecolor='white')
    for i, (cnt, pct) in enumerate(zip(valid_target.values, valid_target / valid_target.sum() * 100)):
        axes[0, 0].text(i, cnt + valid_target.max() * 0.01,
                        f'{pct:.0f}%', ha='center', fontsize=9)
    axes[0, 0].set_title('Phân phối cấp độ kinh nghiệm', fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=30)

    # --- Ô 2: Lương theo cấp độ kinh nghiệm ---
    order = df.groupby(TARGET)[sal_col].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=TARGET, y=sal_col,
                order=order, palette='Set2', ax=axes[0, 1])
    axes[0, 1].set_title(f'{sal_col} theo Cấp độ kinh nghiệm', fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=30)

    # --- Ô 3: Tương quan giữa các cột lương ---
    sal_cols = [c for c in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']
                if c in df.columns]
    if len(sal_cols) >= 2:
        sns.heatmap(df[sal_cols].corr(), annot=True, fmt='.2f', cmap='RdYlGn',
                    center=0, ax=axes[1, 0], linewidths=0.5, annot_kws={'size': 10})
        axes[1, 0].set_title('Tương quan giữa các cột lương', fontweight='bold')

    # --- Ô 4: Quan hệ Lượt xem vs Lượt ứng tuyển ---
    if 'views' in df.columns and 'applies' in df.columns:
        va_df = df[['views', 'applies']].dropna()
        if len(va_df) > 0:
            sample = va_df.sample(min(3000, len(va_df)), random_state=42)
            axes[1, 1].scatter(sample['views'], sample['applies'],
                               alpha=0.3, s=10, color='steelblue')
            r = va_df.corr().iloc[0, 1]
            axes[1, 1].set_title(f'Lượt xem vs Lượt ứng tuyển  (r={r:.2f})', fontweight='bold')
            axes[1, 1].set_xlabel('Lượt xem (Views)')
            axes[1, 1].set_ylabel('Lượt ứng tuyển (Applies)')

    plt.suptitle('Dashboard Tổng kết EDA — DS Job Recommend',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('\nPhân tích EDA hoàn thành!')